# Time Series Data Analysis In Python

Run in a fresh Python 3.11 kernel. Complete original datasets are revision/hash pinned; approved missing-data snapshots come from FRED/BEA and NOAA Central Park. All original gist code is preserved verbatim below. Current pandas/statsmodels APIs, correct market-cap share weighting and actual stock-price labels are maintained. This is a historical data-analysis tutorial, not an investment backtest or a physical climate model. No API keys. All writes use an owned temporary directory.

In [ ]:
import os,sys,subprocess,hashlib,urllib.request,urllib.parse
from pathlib import Path
from tempfile import TemporaryDirectory
subprocess.run([sys.executable,'-m','pip','install','--quiet','--disable-pip-version-check','pandas==3.1.0','numpy==2.3.5','scipy==1.16.3','matplotlib==3.10.8','statsmodels==0.14.6','seaborn==0.13.2','openpyxl==3.1.5','certifi==2026.10.23'],check=True)
import certifi
os.environ['SSL_CERT_FILE']=certifi.where()
os.environ['REQUESTS_CA_BUNDLE']=certifi.where()
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
import seaborn as sns
import statsmodels.api as sm
from IPython.display import display
np.random.seed(42)
pd.set_option('display.max_rows',12)
pd.set_option('display.max_columns',20)
plt.rcParams['figure.dpi']=80
workspace=TemporaryDirectory(prefix='tdb-time-series-')
work=Path(workspace.name)
previous_cwd=Path.cwd()
local_repo=next((p for p in [previous_cwd,*previous_cwd.parents]if(p/'series/time-series-python/data/gdp_growth.csv').exists()),None)
source_revision='65d4ae5d3436be3c0a921a3dcbda1b52e3ac8597'
source_files={'Time Series Manipulation/google.csv': '7bd21a2232a574ef003e1f49d3cfd74dfcd593f0152425164298183476570b24', 'Time Series Manipulation/listings.xlsx': '57496ab13a3f057aa1d4bdd4ca1119344cb71b4790e971c5fc2d8c36384e4195', 'Time Series Manipulation/monthly unemployment rate.csv': 'db39fd7e9e838e44ded9b2a734022a905a5369e45d36b3aa4257081c5f781f68', 'Time Series Manipulation/ozone_nyc.csv': 'eb00aeaa3d4c865c7ea5b19548beed36f647d4c25a608175f7190f19d0294122', 'Time Series Manipulation/price.csv': 'ae42ba97d7c8de1ad994d2d5c2e7629d3208ce12f03af44e9b04555f2aaa8b44', 'Time Series Manipulation/stocks.csv': '6dd14b63c90fdfc847764a75432871ee34151e92e95d8e2c526508d72c31895d', 'Time Series Analysis/DJI.csv': '85782266b0436d5d63e26f8c3f81c698114478914f887b216796d07f00e23dfa', 'Time Series Analysis/HRB.csv': 'd955087c9611805f4f11550b1ec82b68476e38d44fe85e2def389c475d54f22f', 'Time Series Analysis/AMZN.csv': 'cc8e407e621c7f07197bde94aa40552990c81e2fc901d7ef739f5a4c7dc36cdc', 'Time Series Data Visualization/discoveries.csv': '695041e1640165bd7c7794af249a3d77dc7bb6c9cd93e8a4f507e3d4392e2159', 'Time Series Data Visualization/co2_levels.csv': 'ebdff9a6b13b4b1d88c3c3c6dbc7822bf503044322765f91eb36f1137b4e85b5', 'Time Series Data Visualization/airline_passengers.csv': '05ebf0b4a248f489bb046ce45143980ea1bde004790d1a34b9c617df00e4baa3', 'Time Series Data Visualization/meat.csv': 'b6c00e2e82544682dad73a44ffb589f39bb5b97c09b20c68f289a00ad21ebd3f', 'Time Series Data Visualization/employment.csv': 'ba9ee3fb543f2881787a66b63d23ea1f20319f4e09b9db1a1e603753cad96a4a'}
for relative,digest in source_files.items():
    url='https://raw.githubusercontent.com/youssefHosni/Time-Series-Analysis/'+source_revision+'/'+urllib.parse.quote(relative)
    with urllib.request.urlopen(url,timeout=120)as response:payload=response.read()
    assert hashlib.sha256(payload).hexdigest()==digest,relative
    (work/Path(relative).name).write_bytes(payload)
snapshot_hashes={'gdp_growth.csv': '5173321a194285d3958c70e9c7dac9f285414df6087feecbf8aec6f1c109ec2a', 'new_york_temperature.csv': 'a441555f74d927bb1280abade71e63e693a153168bede758c8b844de7a1efa4c'}
for name,digest in snapshot_hashes.items():
    relative='series/time-series-python/data/'+name
    if local_repo:payload=(local_repo/relative).read_bytes()
    else:
        with urllib.request.urlopen('https://raw.githubusercontent.com/To-Data-Beyond/Generative-AI-Techanical-Tutorials/main/'+relative,timeout=120)as response:payload=response.read()
    assert hashlib.sha256(payload).hexdigest()==digest,name
    (work/name).write_bytes(payload)
# The original multi-asset file also supplies the missing asset/SP500 examples.
asset=pd.read_csv(work/'price.csv')
asset.to_csv(work/'asset.csv',index=False)
asset[['DATE','SP500']].rename(columns={'DATE':'date'}).to_csv(work/'sp500.csv',index=False)
(work/'unrate.csv').write_bytes((work/'monthly unemployment rate.csv').read_bytes())
os.chdir(work)
print('Python:',sys.version.split()[0],'| pandas:',pd.__version__,'| NumPy:',np.__version__)
print('14 original datasets verified at pinned revision; NOAA and FRED snapshots verified.')
print('All data writes are in the owned temporary workspace; no API keys.')

In [ ]:
# Compute correlation of levels
# data used is levels
levels = pd.read_csv('DJI.csv', index_col='Date')
levels.index = pd.to_datetime(levels.index.astype(str),format='%Y')

correlation1 = levels['DJI'].corr(levels['UFO'])
print("Correlation of levels: ", correlation1)

# Compute correlation of percent changes
changes = levels.pct_change(fill_method=None)
correlation2 = changes['DJI'].corr(changes['UFO'])
print("Correlation of changes: ", correlation2)
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 1 completed.')

In [ ]:
# Import the statsmodels module
import statsmodels.api as sm

# Compute correlation of x and y

data = pd.read_csv('asset.csv', parse_dates=['DATE'], index_col='DATE')
data = data.dropna()
x = data['SP500']
y = data['Oil']

correlation = x.corr(y)
print("The correlation between x and y is %4.2f" %(correlation))

# Convert the Series x to a DataFrame and name the column x
dfx = pd.DataFrame(x.values, columns=['x'])

# Add a constant to the DataFrame dfx
dfx1 = sm.add_constant(dfx)

# Regress y on dfx1
result = sm.OLS(y.values, dfx1).fit()

# Print out the results and look at the relationship between R-squared and the correlation above
print(result.summary())
ols_result = result
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 2 completed.')

In [ ]:
HRB = pd.read_csv('HRB.csv', index_col='Quarter')
HRB.index = pd.PeriodIndex(HRB.index, freq='Q').to_timestamp()
HRB = HRB.asfreq('QS')
# Import the acf module and the plot_acf module from statsmodels
from statsmodels.tsa.stattools import acf
from statsmodels.graphics.tsaplots import plot_acf

# Compute the acf array of HRB
acf_array = acf(HRB.iloc[:,0])
print(acf_array)

# Plot the acf function
plot_acf(HRB.iloc[:,0], alpha=0.05)
plt.show()
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 3 completed.')

In [ ]:
noise = np.random.normal(loc=0, scale=1, size=500)
plt.plot(noise)
plt.show()
plot_acf(noise, lags=50)
plt.show()
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 4 completed.')

In [ ]:
from statsmodels.tsa.stattools import adfuller
data = pd.read_csv('asset.csv', parse_dates=['DATE'], index_col='DATE')
data = data.dropna()
SPX = data['SP500']
results = adfuller(SPX)
print(results[1]) # print the p-value 
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 5 completed.')

In [ ]:
# Transforming Nonstationary Series Into Stationary Series
fig, axs = plt.subplots(2)
fig.suptitle('Transforming Nonstationary Series Into Stationary Series')
fig.set_size_inches(10, 10.5)

axs[0].plot(HRB)
axs[0].set_title('Quarterly earnings for H&R Block')

axs[1].plot(HRB.diff(4))
display(axs[1].set_title('Quarterly earnings for H&R Block seasonal difference'))
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 6 completed.')

In [ ]:
AMZN = pd.read_csv('AMZN.csv', parse_dates=['Date'], index_col='Date').sort_index()
AMZN = AMZN.resample('QE').last()
fig, axs = plt.subplots(3, figsize=(10,10.5))
axs[0].plot(AMZN);axs[0].set_title('Amazon quarterly adjusted closing stock price')
axs[1].plot(np.log(AMZN));axs[1].set_title('Log quarterly adjusted closing price')
axs[2].plot(np.log(AMZN).diff(4));axs[2].set_title('Four-quarter difference of log price')
plt.tight_layout();plt.show()
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 7 completed.')

In [ ]:
# import the module for simulating data
from statsmodels.tsa.arima_process import ArmaProcess

fig, axs = plt.subplots(4)
fig.set_size_inches(12, 14.5)
fig.suptitle('Simulated data with different AR parameters')

# Plot 1: AR parameter = +0.9
ar1 = np.array([1, -0.9])
ma1 = np.array([1])
AR_object1 = ArmaProcess(ar1, ma1)
simulated_data_1 = AR_object1.generate_sample(nsample=1000)
axs[0].plot(simulated_data_1)
axs[0].set_title('Simulated data with Phi = +0.9 ')

# Plot 2: AR parameter = -0.9
ar2 = np.array([1, 0.9])
ma2 = np.array([1])
AR_object2 = ArmaProcess(ar2, ma2)
simulated_data_2 = AR_object2.generate_sample(nsample=1000)
axs[1].plot(simulated_data_2)
axs[1].set_title('Simulated data with Phi = -0.9 ')


# Plot 3: AR parameter = +0.5
ar3 = np.array([1, -0.5])
ma3 = np.array([1])
AR_object3 = ArmaProcess(ar3, ma3)
simulated_data_3 = AR_object3.generate_sample(nsample=1000)
axs[2].plot(simulated_data_3)
axs[2].set_title('Simulated data with Phi = +0.5')


# Plot 3: AR parameter = -0.5
ar4 = np.array([1, 0.5])
ma4 = np.array([1])
AR_object4 = ArmaProcess(ar4, ma4)
simulated_data_4 = AR_object4.generate_sample(nsample=1000)
axs[3].plot(simulated_data_4)
axs[3].set_title('Simulated data with Phi = -0.5')



original_ar_examples = [simulated_data_1.copy(),simulated_data_2.copy(),simulated_data_3.copy(),simulated_data_4.copy()]
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 8 completed.')

In [ ]:
# Import the plot_acf module from statsmodels
from statsmodels.graphics.tsaplots import plot_acf

fig, axs = plt.subplots(2,2,figsize=(15,10))
fig.suptitle('Autocorrelation functions for different AR parameters')

# Plot 1: AR parameter = +0.9
plot_acf(simulated_data_1 , alpha=None, lags=20, ax=axs[0,0], title='Autocorrelation function for Phi = +0.9')

# Plot 2: AR parameter = -0.9
plot_acf(simulated_data_2 , alpha=None, lags=20, ax=axs[0,1], title='Autocorrelation function for Phi = -0.9')

# Plot 3: AR parameter = +0.5
plot_acf(simulated_data_3, alpha=None, lags=20,  ax=axs[1,0], title='Autocorrelation function for Phi = +0.5')

# Plot 4: AR parameter = -0.5
display(plot_acf(simulated_data_4, alpha=None, lags=20,  ax=axs[1,1], title='Autocorrelation function for Phi = -0.5'))
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 9 completed.')

In [ ]:
simulated_data = simulated_data_1.copy()
from statsmodels.tsa.arima.model import ARIMA
mod = ARIMA(simulated_data, order=(1,0,0))
result = mod.fit(method_kwargs={'maxiter':2000})
print(result.summary())
ar_fit = result
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 10 completed.')

In [ ]:
# Import the modules for simulating data and for plotting the PACF
from statsmodels.tsa.arima_process import ArmaProcess
from statsmodels.graphics.tsaplots import plot_pacf

# Simulate AR(1) with phi=+0.6
ma = np.array([1])
ar = np.array([1, -0.6])
AR_object = ArmaProcess(ar, ma)
simulated_data_1 = AR_object.generate_sample(nsample=5000)

# Plot PACF for AR(1)
plot_pacf(simulated_data_1, lags=20)
plt.show()

# Simulate AR(2) with phi1=+0.6, phi2=+0.3
ma = np.array([1])
ar = np.array([1, -0.6, -0.3])
AR_object = ArmaProcess(ar, ma)
simulated_data_2 = AR_object.generate_sample(nsample=5000)

# Plot PACF for AR(2)
plot_pacf(simulated_data_2, lags=20)
plt.show()
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 11 completed.')

In [ ]:
mod = ARIMA(simulated_data_2, order=(2,0,0))
result = mod.fit(method_kwargs={'maxiter':2000})
print(result.summary())
ar2_fit = result
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 12 completed.')

In [ ]:
bic_fits = []
# Import the module for estimating an ARMA model
from statsmodels.tsa.arima.model import ARIMA

# Fit the data to an AR(p) for p = 0,...,6 , and save the BIC
BIC = np.zeros(7)
for p in range(7):
    mod = ARIMA(simulated_data_2, order=(p,0,0))
    res = mod.fit(method_kwargs={'maxiter':2000})
    bic_fits.append(res)
# Save BIC for AR(p)    
    BIC[p] = res.bic
    
# Plot the BIC as a function of p
plt.plot(range(1,7), BIC[1:7], marker='o')
plt.xlabel('Order of AR Model')
plt.ylabel('Bayesian Information Criterion')
plt.show()
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 13 completed.')

In [ ]:
from statsmodels.tsa.arima_process import ArmaProcess
ar = np.array([1])
ma = np.array([1, 0.5])
AR_object = ArmaProcess(ar, ma)
simulated_data = AR_object.generate_sample(nsample=1000)
plt.plot(simulated_data)
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 14 completed.')

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
mod = ARIMA(simulated_data, order=(0,0,1))
result = mod.fit(method_kwargs={'maxiter':2000})
print(result.summary())
ma_fit = result
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 15 completed.')

In [ ]:
from statsmodels.tsa.stattools import adfuller
temp_NY = pd.read_csv('new_york_temperature.csv', index_col='DATE')
temp_NY.index = pd.to_datetime(temp_NY.index.astype(str),format='%Y')
temp_NY = temp_NY.asfreq('YS')
temp_NY.plot(ylabel='Annual average temperature (°F)');plt.show()
result = adfuller(temp_NY['AVG'])
print('ADF p-value:',result[1])
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 16 completed.')

In [ ]:
# Import the modules for plotting the sample ACF and PACF
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Take first difference of the temperature Series
chg_temp = temp_NY.diff()
chg_temp = chg_temp.dropna()

# Plot the ACF and PACF on the same page
fig, axes = plt.subplots(2,1)

# Plot the ACF
plot_acf(chg_temp, lags=20, ax=axes[0])

# Plot the PACF
plot_pacf(chg_temp, lags=20, ax=axes[1])
plt.show()
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 17 completed.')

In [ ]:
# Import the module for estimating an ARIMA model
from statsmodels.tsa.arima.model import ARIMA

# Fit the data to an AR(1) model and print AIC:
mod_ar1 = ARIMA(chg_temp, order=(1,0,0))
res_ar1 = mod_ar1.fit(method_kwargs={'maxiter':2000})
print("The AIC for an AR(1) is: ", res_ar1.aic)

# Fit the data to an AR(2) model and print AIC:
mod_ar2 = ARIMA(chg_temp, order=(2,0,0))
res_ar2 = mod_ar2.fit(method_kwargs={'maxiter':2000})
print("The AIC for an AR(2) is: ", res_ar2.aic)

# Fit the data to an ARMA(1,1) model and print AIC:
mod_arma11 = ARIMA(chg_temp, order=(1,0,1))
res_arma11 = mod_arma11.fit(method_kwargs={'maxiter':2000})
print("The AIC for an ARMA(1,1) is: ", res_arma11.aic)
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 18 completed.')

In [ ]:
forecast_model = ARIMA(temp_NY['AVG'], order=(1,1,1), trend='t')
forecast_result = forecast_model.fit(method_kwargs={'maxiter':2000})
assert forecast_result.mle_retvals['converged']
forecast = forecast_result.get_forecast(steps=30)
forecast_mean = forecast.predicted_mean
forecast_interval = forecast.conf_int()
ax = temp_NY.plot(ylabel='Annual average temperature (°F)',figsize=(12,6))
forecast_mean.plot(ax=ax,label='30-year ARIMA forecast with drift')
ax.fill_between(forecast_interval.index,forecast_interval.iloc[:,0],forecast_interval.iloc[:,1],alpha=.2)
ax.legend();plt.show()
print('Forecast horizon:',forecast_mean.index[0].year,'to',forecast_mean.index[-1].year)
print('Final forecast (°F):',round(float(forecast_mean.iloc[-1]),6))
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 19 completed.')

In [ ]:
from scipy.signal import lfilter
from numpy.testing import assert_allclose
# OLS reconstructed by NumPy, and the one-regressor/intercept R-squared identity.
design=np.column_stack([np.ones(len(x)),x.to_numpy()])
beta=np.linalg.lstsq(design,y.to_numpy(),rcond=None)[0]
assert_allclose(ols_result.params,beta,rtol=1e-10)
assert_allclose(ols_result.rsquared,float(x.corr(y))**2,rtol=1e-12)
# HRB ACF reconstructed directly using the biased covariance convention.
h=HRB.iloc[:,0].to_numpy();h=h-h.mean()
manual_acf=np.array([np.dot(h[k:],h[:len(h)-k])/np.dot(h,h)for k in range(len(acf_array))])
assert_allclose(acf_array,manual_acf,atol=1e-12)
# Recreate all four full AR simulations from independent filter recurrences.
rng=np.random.RandomState(42);rng.normal(size=500)
for phi,actual in zip([.9,-.9,.5,-.5],original_ar_examples):
    innovations=rng.normal(size=1000)
    expected=lfilter([1.],[1.,-phi],innovations)
    assert_allclose(actual,expected,atol=1e-12)
assert all(fit.mle_retvals['converged']for fit in bic_fits+[ar_fit,ar2_fit,ma_fit,res_ar1,res_ar2,res_arma11,forecast_result])
assert len(BIC)==7 and np.argmin(BIC)==2
assert len(temp_NY)==147 and temp_NY.index[0].year==1870 and temp_NY.index[-1].year==2016
assert len(forecast_mean)==30 and forecast_mean.index[-1].year==2046
assert np.isfinite(forecast_interval.to_numpy()).all()
assert (forecast_interval.iloc[:,0]<forecast_mean).all() and(forecast_mean<forecast_interval.iloc[:,1]).all()
# Independently project all 30 future Kalman state means and covariances.
from scipy.stats import norm
ssm=forecast_result.model.ssm
Z=ssm.design[:,:,0];T=ssm.transition[:,:,0];R=ssm.selection[:,:,0];Q=ssm.state_cov[:,:,0];H=ssm.obs_cov[:,:,0]
state=forecast_result.filter_results.predicted_state[:,-1].copy()
covariance=forecast_result.filter_results.predicted_state_cov[:,:,-1].copy()
expected_mean=[];expected_variance=[]
drift=float(forecast_result.params['x1'])
for horizon in range(30):
    expected_mean.append(float((Z@state)[0]+drift*(forecast_result.nobs+horizon+1)))
    expected_variance.append(float((Z@covariance@Z.T+H)[0,0]))
    state=T@state
    covariance=T@covariance@T.T+R@Q@R.T
assert_allclose(forecast_mean,expected_mean,rtol=1e-10,atol=1e-9)
assert_allclose(forecast.var_pred_mean,expected_variance,rtol=1e-10)
expected_ci=np.column_stack([np.asarray(expected_mean)-norm.ppf(.975)*np.sqrt(expected_variance),np.asarray(expected_mean)+norm.ppf(.975)*np.sqrt(expected_variance)])
assert_allclose(forecast_interval,expected_ci,rtol=1e-10)
# Verify the ADF regression statistic independently with its chosen lag count.
z=temp_NY.AVG.to_numpy();adf=adfuller(z);lag=adf[2];delta=np.diff(z);response=delta[lag:]
columns=[z[lag:-1]]+[delta[lag-k:-k]for k in range(1,lag+1)]+[np.ones(len(response))]
X=np.column_stack(columns);b=np.linalg.lstsq(X,response,rcond=None)[0];residual=response-X@b
variance=np.dot(residual,residual)/(len(response)-X.shape[1]);se=np.sqrt(variance*np.linalg.inv(X.T@X)[0,0])
assert_allclose(b[0]/se,adf[0],rtol=1e-9)
metrics={'ols_correlation':float(x.corr(y)),'ols_r_squared':float(ols_result.rsquared),'ar1_phi':float(ar_fit.arparams[0]),'ar2_phi':ar2_fit.arparams.tolist(),'ma1_theta':float(ma_fit.maparams[0]),'climate_adf_p':float(adf[1]),'climate_aic':[float(res_ar1.aic),float(res_ar2.aic),float(res_arma11.aic)],'forecast_final_fahrenheit':float(forecast_mean.iloc[-1]),'forecast_final_interval':forecast_interval.iloc[-1].tolist()}
print('Independent OLS, ACF, all four AR recurrences, convergence, BIC order and ADF statistic passed.')
print('Full 147-year climate history and 30-year forecast/interval checks passed.')
print('METRICS:',metrics)
os.chdir(previous_cwd)
workspace.cleanup()
print('Owned temporary workspace cleaned up.')

## Original source code

### Original block 1

```text
# Compute correlation of levels
# data used is levels
levels = pd.read_csv('DJI.csv', parse_dates=['Date'], index_col='Date')

correlation1 = levels['DJI'].corr(levels['UFO'])
print("Correlation of levels: ", correlation1)

# Compute correlation of percent changes
changes = levels.pct_change()
correlation2 = changes['DJI'].corr(changes['UFO'])
print("Correlation of changes: ", correlation2)
```

### Original block 2

```text
# Import the statsmodels module
import statsmodels.api as sm

# Compute correlation of x and y

data = pd.read_csv('asset.csv', parse_dates=['DATE'], index_col='DATE')
data = data.dropna()
x = data['SP500']
y = data['Oil']

correlation = x.corr(y)
print("The correlation between x and y is %4.2f" %(correlation))

# Convert the Series x to a DataFrame and name the column x
dfx = pd.DataFrame(x.values, columns=['x'])

# Add a constant to the DataFrame dfx
dfx1 = sm.add_constant(dfx)

# Regress y on dfx1
result = sm.OLS(y.values, dfx1).fit()

# Print out the results and look at the relationship between R-squared and the correlation above
print(result.summary())
```

### Original block 3

```text
# Import the acf module and the plot_acf module from statsmodels
from statsmodels.tsa.stattools import acf
from statsmodels.graphics.tsaplots import plot_acf

# Compute the acf array of HRB
acf_array = acf(HRB)
print(acf_array)

# Plot the acf function
plot_acf(HRB, alpha=0.05)
plt.show()
```

### Original block 4

```text
noise = np.random.normal(loc=0, scale=1, size=500)
plt.plot(noise)
plt.show()
plot_acf(noise, lags=50)
plt.show()
```

### Original block 5

```text
from statsmodels.tsa.stattools import adfuller
data = pd.read_csv('asset.csv', parse_dates=['DATE'], index_col='DATE')
data = data.dropna()
SPX = data['SP500']
results = adfuller(SPX)
print(results[1]) # print the p-value 
```

### Original block 6

```text
# Transforming Nonstationary Series Into Stationary Series
fig, axs = plt.subplots(2)
fig.suptitle('Transforming Nonstationary Series Into Stationary Series')
fig.set_size_inches(10, 10.5)

axs[0].plot(HRB)
axs[0].set_title('Quarterly earnings for H&R Block')

axs[1].plot(HRB.diff(4))
axs[1].set_title('Quarterly earnings for H&R Block seasonal difference')
```

### Original block 7

```text
# Transforming Nonstationary Series Into Stationary Series
AMZN =  pd.read_csv('AMZN.xls', parse_dates=['Date'], index_col='Date')

fig, axs = plt.subplots(3)
fig.suptitle('Transforming Nonstationary Series Into Stationary Series')
fig.set_size_inches(10, 10.5)

axs[0].plot(AMZN)
axs[0].set_title('Amazon quarterly revenue')
axs[1].plot(np.log(AMZN))
axs[1].set_title('Log of Amazon quarterly revenue')

axs[2].plot(np.log(AMZN).diff(4))
axs[2].set_title('Seasonal differnce of Amazon quarterly revenue log')
```

### Original block 8

```text
# import the module for simulating data
from statsmodels.tsa.arima_process import ArmaProcess

fig, axs = plt.subplots(4)
fig.set_size_inches(12, 14.5)
fig.suptitle('Simulated data with different AR parameters')

# Plot 1: AR parameter = +0.9
ar1 = np.array([1, -0.9])
ma1 = np.array([1])
AR_object1 = ArmaProcess(ar1, ma1)
simulated_data_1 = AR_object1.generate_sample(nsample=1000)
axs[0].plot(simulated_data_1)
axs[0].set_title('Simulated data with Phi = +0.9 ')

# Plot 2: AR parameter = -0.9
ar2 = np.array([1, 0.9])
ma2 = np.array([1])
AR_object2 = ArmaProcess(ar2, ma2)
simulated_data_2 = AR_object2.generate_sample(nsample=1000)
axs[1].plot(simulated_data_2)
axs[1].set_title('Simulated data with Phi = -0.9 ')


# Plot 3: AR parameter = +0.5
ar3 = np.array([1, -0.5])
ma3 = np.array([1])
AR_object3 = ArmaProcess(ar3, ma3)
simulated_data_3 = AR_object3.generate_sample(nsample=1000)
axs[2].plot(simulated_data_3)
axs[2].set_title('Simulated data with Phi = +0.5')


# Plot 3: AR parameter = -0.5
ar4 = np.array([1, 0.5])
ma4 = np.array([1])
AR_object4 = ArmaProcess(ar4, ma4)
simulated_data_4 = AR_object4.generate_sample(nsample=1000)
axs[3].plot(simulated_data_4)
axs[3].set_title('Simulated data with Phi = -0.5')



```

### Original block 9

```text
# Import the plot_acf module from statsmodels
from statsmodels.graphics.tsaplots import plot_acf

fig, axs = plt.subplots(2,2,figsize=(15,10))
fig.suptitle('Autocorrelation functions for different AR parameters')

# Plot 1: AR parameter = +0.9
plot_acf(simulated_data_1 , alpha=1, lags=20, ax=axs[0,0], title='Autocorrelation function for Phi = +0.9')

# Plot 2: AR parameter = -0.9
plot_acf(simulated_data_2 , alpha=1, lags=20, ax=axs[0,1], title='Autocorrelation function for Phi = -0.9')

# Plot 3: AR parameter = +0.5
plot_acf(simulated_data_3, alpha=1, lags=20,  ax=axs[1,0], title='Autocorrelation function for Phi = +0.5')

# Plot 4: AR parameter = -0.5
plot_acf(simulated_data_4, alpha=1, lags=20,  ax=axs[1,1], title='Autocorrelation function for Phi = -0.5')

```

### Original block 10

```text
from statsmodels.tsa.arima.model import ARIMA
mod = ARIMA(simulated_data, order=(1,0,0))
result = mod.fit()
print(result.summary())
```

### Original block 11

```text
# Import the modules for simulating data and for plotting the PACF
from statsmodels.tsa.arima_process import ArmaProcess
from statsmodels.graphics.tsaplots import plot_pacf

# Simulate AR(1) with phi=+0.6
ma = np.array([1])
ar = np.array([1, -0.6])
AR_object = ArmaProcess(ar, ma)
simulated_data_1 = AR_object.generate_sample(nsample=5000)

# Plot PACF for AR(1)
plot_pacf(simulated_data_1, lags=20)
plt.show()

# Simulate AR(2) with phi1=+0.6, phi2=+0.3
ma = np.array([1])
ar = np.array([1, -0.6, -0.3])
AR_object = ArmaProcess(ar, ma)
simulated_data_2 = AR_object.generate_sample(nsample=5000)

# Plot PACF for AR(2)
plot_pacf(simulated_data_2, lags=20)
plt.show()
```

### Original block 12

```text
mod = ARIMA(simulated_data_2, order=(2,0,0))
result = mod.fit()
print(result.summary())
```

### Original block 13

```text
# Import the module for estimating an ARMA model
from statsmodels.tsa.arima.model import ARIMA

# Fit the data to an AR(p) for p = 0,...,6 , and save the BIC
BIC = np.zeros(7)
for p in range(7):
    mod = ARIMA(simulated_data_2, order=(p,0,0))
    res = mod.fit()
# Save BIC for AR(p)    
    BIC[p] = res.bic
    
# Plot the BIC as a function of p
plt.plot(range(1,7), BIC[1:7], marker='o')
plt.xlabel('Order of AR Model')
plt.ylabel('Bayesian Information Criterion')
plt.show()
```

### Original block 14

```text
from statsmodels.tsa.arima_process import ArmaProcess
ar = np.array([1])
ma = np.array([1, 0.5])
AR_object = ArmaProcess(ar, ma)
simulated_data = AR_object.generate_sample(nsample=1000)
plt.plot(simulated_data)
```

### Original block 15

```text
from statsmodels.tsa.arima.model import ARIMA
mod = ARIMA(simulated_data, order=(0,0,1))
result = mod.fit()
print(result.summary())
```

### Original block 16

```text
# Import the adfuller function from the statsmodels module
from statsmodels.tsa.stattools import adfuller

#upload the data 
temp_NY = pd.read_csv('New York temperature data.csv', parse_dates=['DATE'], index_col='DATE' )

# Convert the index to a datetime object
temp_NY.index = pd.to_datetime(temp_NY.index, format='%Y')

# Plot average temperatures
temp_NY.plot()
plt.show()

# Compute and print ADF p-value
result = adfuller(temp_NY['AVG'])
print("The p-value for the ADF test is ", result[1])
```

### Original block 17

```text
# Import the modules for plotting the sample ACF and PACF
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Take first difference of the temperature Series
chg_temp = temp_NY.diff()
chg_temp = chg_temp.dropna()

# Plot the ACF and PACF on the same page
fig, axes = plt.subplots(2,1)

# Plot the ACF
plot_acf(chg_temp, lags=20, ax=axes[0])

# Plot the PACF
plot_pacf(chg_temp, lags=20, ax=axes[1])
plt.show()
```

### Original block 18

```text
# Import the module for estimating an ARIMA model
from statsmodels.tsa.arima.model import ARIMA

# Fit the data to an AR(1) model and print AIC:
mod_ar1 = ARIMA(chg_temp, order=(1,0,0))
res_ar1 = mod_ar1.fit()
print("The AIC for an AR(1) is: ", res_ar1.aic)

# Fit the data to an AR(2) model and print AIC:
mod_ar2 = ARIMA(chg_temp, order=(2,0,0))
res_ar2 = mod_ar2.fit()
print("The AIC for an AR(2) is: ", res_ar2.aic)

# Fit the data to an ARMA(1,1) model and print AIC:
mod_arma11 = ARIMA(chg_temp, order=(1,0,1))
res_arma11 = mod_arma11.fit()
print("The AIC for an ARMA(1,1) is: ", res_arma11.aic)
```